
# XGBoost–GRU Ensemble for TwinSepsis

## Objective
Combine XGBoost and GRU hourly sepsis probabilities using a weighted probability ensemble.

## Methodology
1. Align validation predictions by patient and ICU hour.
2. Select ensemble weights using validation data.
3. Select the classification threshold using validation data.
4. Compare against individual models.
5. Perform final test evaluation after model selection.


In [14]:

# Load saved validation predictions; no model retraining required

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

RESULTS_DIR = Path(r"C:\TwinSepsis\results")

gru = pd.read_csv(RESULTS_DIR / "gru_validation_predictions.csv")
xgb = pd.read_csv(RESULTS_DIR / "xgboost_validation_predictions.csv")

print("GRU rows:", len(gru))
print("XGBoost rows:", len(xgb))


GRU rows: 231472
XGBoost rows: 231472


In [15]:

# Cell 2: Align GRU and XGBoost predictions by patient and hour

gru = pd.read_csv(RESULTS_DIR / "gru_validation_predictions.csv")
xgb = pd.read_csv(RESULTS_DIR / "xgboost_validation_predictions.csv")

merged = gru.merge(
    xgb,
    on=["patient_id", "hour_index"],
    how="outer",
    suffixes=("_gru", "_xgb"),
    indicator=True,
    validate="one_to_one",
)

print("GRU rows:", len(gru))
print("XGBoost rows:", len(xgb))
print("Merged rows:", len(merged))
print("\nMerge status:")
print(merged["_merge"].value_counts())

matched = merged[merged["_merge"] == "both"].copy()

labels_match = (
    matched["SepsisLabel_gru"] == matched["SepsisLabel_xgb"]
).all()

print("\nAll patient-hour labels match:", labels_match)

if len(matched) != len(gru) or len(matched) != len(xgb):
    raise ValueError("Patient-hour alignment failed.")

if not labels_match:
    raise ValueError("The two models have different labels.")

ensemble_val = matched[
    [
        "patient_id",
        "hour_index",
        "SepsisLabel_gru",
        "gru_probability",
        "xgb_probability",
    ]
].rename(columns={"SepsisLabel_gru": "SepsisLabel"}).copy()

print("\nAlignment verified.")
print("Aligned rows:", len(ensemble_val))
print("Positive hours:", int(ensemble_val["SepsisLabel"].sum()))

display(ensemble_val.head())


GRU rows: 231472
XGBoost rows: 231472
Merged rows: 231472

Merge status:
_merge
both          231472
left_only          0
right_only         0
Name: count, dtype: int64

All patient-hour labels match: True

Alignment verified.
Aligned rows: 231472
Positive hours: 4162


,patient_id,hour_index,SepsisLabel,gru_probability,xgb_probability
0,p000009,0,0,0.067594,0.387668
1,p000009,1,0,0.040482,0.579060
2,p000009,2,0,0.079162,0.430713
3,p000009,3,0,0.115043,0.511216
4,p000009,4,0,0.172759,0.572414


In [16]:

# Cell 3: Validation-only ensemble weight search

y_val = ensemble_val["SepsisLabel"].to_numpy()
gru_prob = ensemble_val["gru_probability"].to_numpy()
xgb_prob = ensemble_val["xgb_probability"].to_numpy()

weight_rows = []

for w in np.round(np.arange(0, 1.001, 0.05), 2):
    # w = XGBoost weight; 1-w = GRU weight
    ensemble_prob = w * xgb_prob + (1 - w) * gru_prob

    weight_rows.append({
        "xgb_weight": w,
        "gru_weight": round(1 - w, 2),
        "ROC_AUC": roc_auc_score(y_val, ensemble_prob),
        "AUPRC": average_precision_score(y_val, ensemble_prob),
    })

weight_results = pd.DataFrame(weight_rows)

# Include individual models explicitly for comparison
individual_results = pd.DataFrame([
    {
        "xgb_weight": 1.0,
        "gru_weight": 0.0,
        "ROC_AUC": roc_auc_score(y_val, xgb_prob),
        "AUPRC": average_precision_score(y_val, xgb_prob),
    },
    {
        "xgb_weight": 0.0,
        "gru_weight": 1.0,
        "ROC_AUC": roc_auc_score(y_val, gru_prob),
        "AUPRC": average_precision_score(y_val, gru_prob),
    },
])

weight_results = (
    pd.concat([weight_results, individual_results])
    .drop_duplicates(["xgb_weight", "gru_weight"])
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("Top 10 configurations by validation ROC-AUC")
display(weight_results.head(10))

print("\nBest ROC-AUC configuration:")
display(weight_results.iloc[[0]])


Top 10 configurations by validation ROC-AUC


,xgb_weight,gru_weight,ROC_AUC,AUPRC
0,0.55,0.45,0.812665,0.085824
1,0.60,0.40,0.812551,0.086024
2,0.50,0.50,0.812526,0.085511
3,0.45,0.55,0.812166,0.085133
4,0.65,0.35,0.812137,0.086247
5,0.40,0.60,0.811588,0.084689
6,0.70,0.30,0.811361,0.086273
7,0.35,0.65,0.810792,0.084365
8,0.75,0.25,0.810155,0.086104
9,0.30,0.70,0.809737,0.083818



Best ROC-AUC configuration:


,xgb_weight,gru_weight,ROC_AUC,AUPRC
0,0.55,0.45,0.812665,0.085824


In [17]:

# Cell 4: Record the selected blend and compare validation performance

best_row = weight_results.iloc[0]
best_xgb_weight = float(best_row["xgb_weight"])
best_gru_weight = float(best_row["gru_weight"])

best_ensemble_prob = (
    best_xgb_weight * xgb_prob
    + best_gru_weight * gru_prob
)

comparison = pd.DataFrame([
    {
        "Model": "XGBoost",
        "ROC_AUC": roc_auc_score(y_val, xgb_prob),
        "AUPRC": average_precision_score(y_val, xgb_prob),
    },
    {
        "Model": "GRU",
        "ROC_AUC": roc_auc_score(y_val, gru_prob),
        "AUPRC": average_precision_score(y_val, gru_prob),
    },
    {
        "Model": "Weighted Ensemble",
        "ROC_AUC": roc_auc_score(y_val, best_ensemble_prob),
        "AUPRC": average_precision_score(y_val, best_ensemble_prob),
    },
])

print(f"Selected XGBoost weight: {best_xgb_weight:.2f}")
print(f"Selected GRU weight: {best_gru_weight:.2f}")
display(comparison)

comparison.to_csv(
    RESULTS_DIR / "ensemble_validation_comparison.csv",
    index=False,
)
weight_results.to_csv(
    RESULTS_DIR / "ensemble_validation_weight_search.csv",
    index=False,
)

print("Validation comparison and weight search saved.")


Selected XGBoost weight: 0.55
Selected GRU weight: 0.45


,Model,ROC_AUC,AUPRC
0,XGBoost,0.796669,0.082692
1,GRU,0.789997,0.074066
2,Weighted Ensemble,0.812665,0.085824


Validation comparison and weight search saved.


In [18]:

# Cell 5: Choose a classification threshold using validation F1

from sklearn.metrics import precision_score, recall_score, f1_score

threshold_rows = []

for threshold in np.round(np.arange(0.01, 0.951, 0.01), 2):
    pred = (best_ensemble_prob >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_threshold_row["threshold"])

print("Best validation threshold:")
display(best_threshold_row.to_frame().T)

print("\nTop 10 thresholds by validation F1:")
display(
    threshold_results.sort_values("f1", ascending=False).head(10)
)

threshold_results.to_csv(
    RESULTS_DIR / "ensemble_validation_threshold_search.csv",
    index=False,
)


Best validation threshold:


,threshold,precision,recall,f1
80,0.81,0.11037,0.282316,0.158698



Top 10 thresholds by validation F1:


,threshold,precision,recall,f1
80,0.81,0.110370,0.282316,0.158698
85,0.86,0.126653,0.211677,0.158482
84,0.85,0.120602,0.228976,0.157991
79,0.80,0.108024,0.290485,0.157483
78,0.79,0.106854,0.298895,0.157428
76,0.77,0.104734,0.315233,0.157229
75,0.76,0.103850,0.323402,0.157215
81,0.82,0.110913,0.268621,0.157000
83,0.84,0.115780,0.243633,0.156966
77,0.78,0.105503,0.306343,0.156952


In [19]:

# Cell 6: Compare threshold-based validation metrics

def threshold_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "threshold": threshold,
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
    }

threshold_comparison = pd.DataFrame([
    {
        "Model": "XGBoost",
        **threshold_metrics(y_val, xgb_prob, best_threshold),
    },
    {
        "Model": "GRU",
        **threshold_metrics(y_val, gru_prob, best_threshold),
    },
    {
        "Model": "Weighted Ensemble",
        **threshold_metrics(y_val, best_ensemble_prob, best_threshold),
    },
])

display(threshold_comparison)

threshold_comparison.to_csv(
    RESULTS_DIR / "ensemble_validation_threshold_comparison.csv",
    index=False,
)


,Model,threshold,precision,recall,f1
0,XGBoost,0.81,0.114625,0.243873,0.155950
1,GRU,0.81,0.087849,0.358722,0.141135
2,Weighted Ensemble,0.81,0.110370,0.282316,0.158698


In [20]:

# Cell 7: Save the selected ensemble configuration

import json

ensemble_config = {
    "model": "weighted_probability_ensemble",
    "components": ["XGBoost", "GRU"],
    "xgboost_weight": best_xgb_weight,
    "gru_weight": best_gru_weight,
    "selection_metric": "validation_ROC_AUC",
    "threshold": best_threshold,
    "threshold_selection_metric": "validation_F1",
    "validation_rows": int(len(ensemble_val)),
    "validation_patients": int(ensemble_val["patient_id"].nunique()),
    "test_evaluated": False,
}

config_path = RESULTS_DIR / "ensemble_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(ensemble_config, f, indent=2)

print("Saved:", config_path)
print(json.dumps(ensemble_config, indent=2))


Saved: C:\TwinSepsis\results\ensemble_config.json
{
  "model": "weighted_probability_ensemble",
  "components": [
    "XGBoost",
    "GRU"
  ],
  "xgboost_weight": 0.55,
  "gru_weight": 0.45,
  "selection_metric": "validation_ROC_AUC",
  "threshold": 0.81,
  "threshold_selection_metric": "validation_F1",
  "validation_rows": 231472,
  "validation_patients": 6050,
  "test_evaluated": false
}



## Final Test Evaluation

The ensemble weights and classification threshold have been selected using validation data only. The held-out test set will now be used for one final evaluation without further model or threshold tuning.

The evaluation will report ROC-AUC, AUPRC, precision, recall, F1-score, and the confusion matrix for XGBoost, GRU, and the weighted ensemble.


In [22]:

# Final test evaluation: XGBoost vs GRU vs weighted ensemble
# Uses frozen validation-selected weights and threshold.
# Does not retrain either model.

import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.nn.utils.rnn import pad_sequence
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Paths and frozen validation-selected configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\TwinSepsis")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TRAIN_A, TRAIN_B, RESULTS_DIR, MODELS_DIR

with open(RESULTS_DIR / "ensemble_config.json", encoding="utf-8") as f:
    config = json.load(f)

XGB_WEIGHT = float(config["xgboost_weight"])
GRU_WEIGHT = float(config["gru_weight"])
THRESHOLD = float(config["threshold"])

assert np.isclose(XGB_WEIGHT + GRU_WEIGHT, 1.0)

print("XGBoost weight:", XGB_WEIGHT)
print("GRU weight:", GRU_WEIGHT)
print("Frozen threshold:", THRESHOLD)

# ------------------------------------------------------------
# 2. Load test patient IDs and saved checkpoints
# ------------------------------------------------------------

test_ids = pd.read_csv(
    RESULTS_DIR / "test_patient_ids.csv",
    dtype={"patient_id": str}
)["patient_id"].tolist()

assert len(test_ids) == len(set(test_ids)), (
    "Duplicate patient IDs found in the test split."
)

all_patient_files = {}

for folder in [TRAIN_A, TRAIN_B]:
    for path in folder.glob("*.psv"):
        all_patient_files[path.stem] = path

missing = [pid for pid in test_ids if pid not in all_patient_files]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} test patient files are missing. "
        f"Examples: {missing[:5]}"
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference device:", device)
print("Test patients:", len(test_ids))

gru_checkpoint = torch.load(
    MODELS_DIR / "gru_baseline.pt",
    map_location=device,
    weights_only=False,
)
feature_columns = list(gru_checkpoint["feature_columns"])
measured_columns = list(gru_checkpoint["measured_columns"])

medians = np.asarray(gru_checkpoint["medians"], dtype=np.float32)
means = np.asarray(gru_checkpoint["means"], dtype=np.float32)
stds = np.asarray(gru_checkpoint["stds"], dtype=np.float32)
clip_value = float(gru_checkpoint["clip"])

measured_indices = [
    feature_columns.index(col) for col in measured_columns
]

# ------------------------------------------------------------
# 3. Define the original GRU architecture
# ------------------------------------------------------------

class SepsisGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64,
                 num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out).squeeze(-1)

gru_model = SepsisGRU(**gru_checkpoint["model_config"]).to(device)
gru_model.load_state_dict(gru_checkpoint["model_state_dict"])
gru_model.eval()

# ------------------------------------------------------------
# 4. Load test patients and apply the original preprocessing
# ------------------------------------------------------------

test_frames = []
gru_test_data = {}

for i, pid in enumerate(test_ids):
    df = pd.read_csv(all_patient_files[pid], sep="|")

    # Retain patient/hour identifiers and the target
    y = df["SepsisLabel"].to_numpy(dtype=np.int8)
    X = (
        df[feature_columns]
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float32)
    )

    # XGBoost uses the original 40 raw input features
    xgb_frame = df[feature_columns].copy()
    xgb_frame["patient_id"] = pid
    xgb_frame["hour_index"] = np.arange(len(df))
    xgb_frame["SepsisLabel"] = y
    test_frames.append(xgb_frame)

    # GRU preprocessing: missingness flags, forward fill,
    # training medians, training means/stds, and clipping
    observed = (~np.isnan(X[:, measured_indices])).astype(np.float32)

    X_filled = (
        pd.DataFrame(X)
        .ffill()
        .to_numpy(dtype=np.float32, copy=True)
    )

    nan_mask = np.isnan(X_filled)

    X_filled[nan_mask] = np.take(
        medians,
        np.where(nan_mask)[1]
    )

    X_scaled = np.clip(
        (X_filled - means) / stds,
        -clip_value,
        clip_value,
    )

    X_gru = np.concatenate(
        [X_scaled, observed], axis=1
    ).astype(np.float32)

    gru_test_data[pid] = (X_gru, y)

    if (i + 1) % 1000 == 0:
        print(f"Prepared {i + 1:,}/{len(test_ids):,} patients")

test_df = pd.concat(test_frames, ignore_index=True)

print("\nTest hourly rows:", len(test_df))
print("Test positive hours:", int(test_df["SepsisLabel"].sum()))

# ------------------------------------------------------------
# 5. Generate XGBoost test probabilities
# ------------------------------------------------------------

xgb_model = XGBClassifier()
xgb_model.load_model(MODELS_DIR / "xgboost_baseline.json")

# Verify XGBoost and GRU feature order before inference

xgb_feature_names = xgb_model.get_booster().feature_names

if xgb_feature_names is not None:
    if list(xgb_feature_names) != feature_columns:
        raise ValueError(
            "Feature names or order differ between XGBoost "
            "and the GRU checkpoint."
        )
if xgb_model.n_features_in_ != len(feature_columns):
    raise ValueError(
        f"XGBoost expects {xgb_model.n_features_in_} features, "
        f"but the GRU checkpoint provides {len(feature_columns)}."
    )

print("Feature-order check passed.")
print("Number of input features:", len(feature_columns))

X_test_xgb = (
    test_df[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

xgb_test_prob = xgb_model.predict_proba(X_test_xgb)[:, 1]

xgb_test_export = pd.DataFrame({
    "patient_id": test_df["patient_id"].astype(str),
    "hour_index": test_df["hour_index"].astype(int),
    "SepsisLabel": test_df["SepsisLabel"].astype(int),
    "xgb_probability": xgb_test_prob,
})

# ------------------------------------------------------------
# 6. Generate GRU test probabilities by patient
# ------------------------------------------------------------

gru_test_rows = []
batch_size = 128
ordered_test_ids = list(test_ids)

with torch.no_grad():
    for start in range(0, len(ordered_test_ids), batch_size):
        batch_ids = ordered_test_ids[start:start + batch_size]
        sequences = [
            torch.from_numpy(gru_test_data[pid][0])
            for pid in batch_ids
        ]

        lengths = [len(seq) for seq in sequences]
        padded = pad_sequence(sequences, batch_first=True)

        logits = gru_model(padded.to(device)).float().cpu()
        probabilities = torch.sigmoid(logits).numpy()

        for row, pid in enumerate(batch_ids):
            labels = gru_test_data[pid][1]
            probs = probabilities[row, :lengths[row]]

            for hour_idx, (label, prob) in enumerate(zip(labels, probs)):
                gru_test_rows.append({
                    "patient_id": pid,
                    "hour_index": hour_idx,
                    "SepsisLabel": int(label),
                    "gru_probability": float(prob),
                })

    print(
        f"Processed {min(start + batch_size, len(ordered_test_ids)):,}"
        f"/{len(ordered_test_ids):,} patients"
    )

gru_test_export = pd.DataFrame(gru_test_rows)

# ------------------------------------------------------------
# 7. Align test predictions and verify labels
# ------------------------------------------------------------

test_merged = gru_test_export.merge(
    xgb_test_export,
    on=["patient_id", "hour_index"],
    how="outer",
    suffixes=("_gru", "_xgb"),
    indicator=True,
    validate="one_to_one",
)

print("\nTest merge status:")
print(test_merged["_merge"].value_counts())

matched = test_merged[test_merged["_merge"] == "both"].copy()

if len(matched) != len(gru_test_export) or len(matched) != len(xgb_test_export):
    raise ValueError("GRU and XGBoost test rows do not align.")

if not (
    matched["SepsisLabel_gru"] == matched["SepsisLabel_xgb"]
).all():
    raise ValueError("GRU and XGBoost test labels differ.")

y_test = matched["SepsisLabel_gru"].to_numpy(dtype=int)
gru_prob = matched["gru_probability"].to_numpy()
xgb_prob = matched["xgb_probability"].to_numpy()

ensemble_prob = XGB_WEIGHT * xgb_prob + GRU_WEIGHT * gru_prob

print("Aligned test rows:", len(matched))
print("Test positive hours:", int(y_test.sum()))
print("Test labels agree: True")

# ------------------------------------------------------------
# 8. Final evaluation at the frozen threshold
# ------------------------------------------------------------

def calculate_metrics(name, y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "model": name,
        "threshold": threshold,
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "AUPRC": average_precision_score(y_true, probabilities),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "test_rows": len(y_true),
        "positive_hours": int(y_true.sum()),
    }

test_results = pd.DataFrame([
    calculate_metrics("XGBoost", y_test, xgb_prob, THRESHOLD),
    calculate_metrics("GRU", y_test, gru_prob, THRESHOLD),
    calculate_metrics("Weighted Ensemble", y_test, ensemble_prob, THRESHOLD),
])

print("\nFINAL TEST RESULTS")
display(test_results)

test_results.to_csv(
    RESULTS_DIR / "ensemble_final_test_comparison.csv",
    index=False,
)

# Save hourly probabilities for reproducibility
final_predictions = pd.DataFrame({
    "patient_id": matched["patient_id"].astype(str),
    "hour_index": matched["hour_index"].astype(int),
    "SepsisLabel": y_test,
    "xgb_probability": xgb_prob,
    "gru_probability": gru_prob,
    "ensemble_probability": ensemble_prob,
})

final_predictions.to_csv(
    RESULTS_DIR / "ensemble_test_predictions.csv",
    index=False,
)

# Save confusion matrices
for name, probabilities in [
    ("xgboost", xgb_prob),
    ("gru", gru_prob),
    ("ensemble", ensemble_prob),
]:
    predictions = (probabilities >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    pd.DataFrame(
        cm,
        index=["Actual_No_Sepsis", "Actual_Sepsis"],
        columns=["Predicted_No_Sepsis", "Predicted_Sepsis"],
    ).to_csv(RESULTS_DIR / f"{name}_ensemble_test_confusion_matrix.csv")

print("\nSaved final test comparison and hourly predictions.")
print("Test evaluation complete; do not retune based on these results.")


XGBoost weight: 0.55
GRU weight: 0.45
Frozen threshold: 0.81
Inference device: cpu
Test patients: 6051
Prepared 1,000/6,051 patients
Prepared 2,000/6,051 patients
Prepared 3,000/6,051 patients
Prepared 4,000/6,051 patients
Prepared 5,000/6,051 patients
Prepared 6,000/6,051 patients

Test hourly rows: 234302
Test positive hours: 4194
Feature-order check passed.
Number of input features: 40
Processed 6,051/6,051 patients

Test merge status:
_merge
both          234302
left_only          0
right_only         0
Name: count, dtype: int64
Aligned test rows: 234302
Test positive hours: 4194
Test labels agree: True

FINAL TEST RESULTS


,model,threshold,ROC_AUC,AUPRC,accuracy,precision,recall,F1,test_rows,positive_hours
0,XGBoost,0.81,0.816563,0.121752,0.952924,0.139680,0.315928,0.193713,234302,4194
1,GRU,0.81,0.815660,0.101831,0.921644,0.101547,0.430377,0.164322,234302,4194
2,Weighted Ensemble,0.81,0.838431,0.131937,0.946065,0.128814,0.349309,0.188219,234302,4194



Saved final test comparison and hourly predictions.
Test evaluation complete; do not retune based on these results.
